# Hypothesis Test Decision Guide for Industry Problems

This notebook is a practical guide for choosing the right hypothesis test in
business and data science projects.

It is based on the tests already covered in the previous hypothesis testing
notebooks:

| Covered Area | Tests |
| --- | --- |
| Mean comparison | z-test, one-sample t-test, paired t-test, independent t-test |
| Multi-group mean comparison | one-way ANOVA, two-way ANOVA, ANCOVA |
| Non-parametric comparison | Wilcoxon, Mann-Whitney U, Friedman, Kruskal-Wallis, Mood's Median, Brunner-Munzel |
| Categorical testing | chi-square, Fisher's exact, McNemar, Cochran's Q |
| Proportion testing | one-proportion z-test |
| Assumption testing | Shapiro-Wilk, Kolmogorov-Smirnov, Anderson-Darling, D'Agostino-Pearson, Jarque-Bera |
| Variance / spread testing | Levene, Bartlett, Fligner-Killeen, Ansari-Bradley |
| Relationship testing | Kendall Tau |
| Process / data quality | Runs test, Grubbs test |
| Post-hoc comparison | Tukey HSD |

The goal is not to memorize tests. The goal is to ask the correct business and
data questions, then choose the test logically.

## The Golden Rule

Before selecting any hypothesis test, answer these 7 questions:

| Question | Why it matters |
| --- | --- |
| 1. What is the business question? | The test must answer the business decision, not just produce a p-value. |
| 2. What type is the target variable? | Numeric, categorical, binary, ordinal, or time-ordered. |
| 3. How many groups are being compared? | One, two, or more than two groups changes the test. |
| 4. Are the samples independent or paired? | Same customers/employees before-after means paired. Different customers means independent. |
| 5. Is the data approximately normal? | Normal data allows parametric tests; non-normal data often needs non-parametric tests. |
| 6. Are variances similar across groups? | Unequal variance can change the test choice. |
| 7. Is the sample size small? | Small categorical samples may need exact tests like Fisher's exact test. |

## Simple Decision Flowchart

Use this as a first-pass flowchart.

```text
START
 |
 |-- Are you checking data assumptions?
 |      |-- Normality? ----------> Shapiro-Wilk / KS / Anderson-Darling / D'Agostino / Jarque-Bera
 |      |-- Equal variance? -----> Levene / Bartlett / Fligner-Killeen
 |      |-- Outlier? ------------> Grubbs Test
 |      |-- Random sequence? ----> Runs Test
 |
 |-- Is your outcome numeric?
 |      |-- Comparing to one fixed value?
 |      |       |-- Population std known / large n --> z-test
 |      |       |-- Population std unknown --------> one-sample t-test
 |      |
 |      |-- Comparing two groups?
 |      |       |-- Same people/items? 
 |      |       |       |-- Normal differences ------> paired t-test
 |      |       |       |-- Non-normal differences --> Wilcoxon signed-rank
 |      |       |
 |      |       |-- Different people/items?
 |      |               |-- Normal data -------------> independent t-test
 |      |               |-- Non-normal data ---------> Mann-Whitney U / Brunner-Munzel
 |      |
 |      |-- Comparing more than two groups?
 |              |-- Same people/items?
 |              |       |-- Normal data -------------> repeated-measures ANOVA
 |              |       |-- Non-normal data ---------> Friedman test
 |              |
 |              |-- Different people/items?
 |                      |-- Normal data -------------> one-way ANOVA
 |                      |-- Non-normal data ---------> Kruskal-Wallis / Mood's Median
 |
 |-- Is your outcome categorical or binary?
 |      |-- One categorical variable vs expected distribution? --> Chi-square goodness-of-fit
 |      |-- Two independent categorical variables? -------------> Chi-square independence
 |      |-- Small 2 x 2 table? ---------------------------------> Fisher's exact
 |      |-- Same subjects, before-after binary? ----------------> McNemar
 |      |-- Same subjects, more than two binary treatments? -----> Cochran's Q
 |      |-- One binary proportion vs claimed value? ------------> one-proportion z-test
 |
 |-- Are you studying relationship?
        |-- Ranked / ordinal variables? ------------------------> Kendall Tau
        |-- Numeric relationship with covariate adjustment? ----> ANCOVA / regression-style analysis
```

## Master Business Decision Table

This table is the fastest way to choose a test in an industry project.

In [1]:
import pandas as pd

decision_table = pd.DataFrame([
    ["Average order value changed from target", "Numeric", "One sample vs fixed value", "Independent", "Approximately normal", "one-sample t-test", "Use z-test if population std is known or sample is very large"],
    ["Average IQ / payload / sales vs known claim", "Numeric", "One sample vs fixed value", "Independent", "Population std known", "z-test", "Usually used in textbook/business claim problems"],
    ["Employee score before vs after training", "Numeric", "Two related measurements", "Paired", "Differences normal", "paired t-test", "Same people measured twice"],
    ["Calcium level before vs after treatment", "Numeric", "Two related measurements", "Paired", "Differences non-normal", "Wilcoxon signed-rank", "Non-parametric alternative to paired t-test"],
    ["Sales of two independent regions", "Numeric", "Two groups", "Independent", "Normal", "independent t-test", "Use Welch version if variances are unequal"],
    ["Ratings of two app versions", "Ordinal/Numeric", "Two groups", "Independent", "Non-normal", "Mann-Whitney U", "Use Brunner-Munzel if shapes/variances differ strongly"],
    ["Stress levels across three departments", "Numeric", "More than two groups", "Independent", "Normal", "one-way ANOVA", "Follow with Tukey HSD if ANOVA is significant"],
    ["Customer spend across three stores", "Skewed numeric", "More than two groups", "Independent", "Non-normal", "Kruskal-Wallis or Mood's Median", "Kruskal compares ranks; Mood focuses on medians"],
    ["Same patients measured at three time points", "Numeric", "More than two related measurements", "Paired", "Non-normal", "Friedman test", "Non-parametric repeated-measures test"],
    ["Effect of city and offer type on sales", "Numeric", "Two categorical factors", "Independent", "Normal", "two-way ANOVA", "Tests main effects and interaction"],
    ["Compare groups while controlling age/income", "Numeric", "Group comparison with covariate", "Independent", "Model-based", "ANCOVA", "Useful when a continuous variable affects outcome"],
    ["Region related to conversion status", "Categorical", "Two categorical variables", "Independent", "Expected counts adequate", "chi-square independence", "For association between categories"],
    ["Observed dice/web traffic vs expected split", "Categorical counts", "One variable vs expected distribution", "Independent", "Expected counts adequate", "chi-square goodness-of-fit", "Checks if actual distribution matches expected"],
    ["Small treatment success table", "Binary categorical", "2 x 2 table", "Independent", "Small sample", "Fisher's exact", "Exact alternative to chi-square for small samples"],
    ["Old model vs new model on same records", "Binary correct/incorrect", "Two related classifiers", "Paired", "Binary", "McNemar", "Best for paired classifier comparison"],
    ["Same customers click Ad A/B/C", "Binary", "More than two related treatments", "Paired", "Binary", "Cochran's Q", "Extension of McNemar for 3+ related binary conditions"],
    ["Conversion rate vs claimed conversion", "Binary proportion", "One sample proportion", "Independent", "Large enough n", "one-proportion z-test", "For click/churn/defect/conversion rate claims"],
    ["Satisfaction rank vs renewal rank", "Ordinal/ranked", "Relationship", "Same rows", "Ranked data", "Kendall Tau", "For ordinal association"],
    ["Check if data is normal before parametric test", "Numeric", "Assumption check", "Any", "Normality assumption", "Shapiro-Wilk / KS / Anderson-Darling", "Use visual checks also"],
    ["Check if residuals have normal skew/kurtosis", "Numeric", "Assumption check", "Any", "Normality of errors", "D'Agostino-Pearson / Jarque-Bera", "Common in modeling diagnostics"],
    ["Check if city delivery variance differs", "Numeric", "Spread comparison", "Independent groups", "Robust", "Levene / Fligner-Killeen", "Fligner is safer for skewed data"],
    ["Check machine measurement variance", "Numeric", "Spread comparison", "Independent groups", "Normal", "Bartlett", "Powerful but sensitive to non-normality"],
    ["Check if one sensor is more variable", "Numeric", "Two group spread", "Independent", "Non-parametric", "Ansari-Bradley", "Compares scale/spread"],
    ["Check whether SLA failures cluster over time", "Binary sequence", "Randomness", "Time-ordered", "Sequence", "Runs test", "Detects clustering or non-random pattern"],
    ["Check one suspicious manufacturing reading", "Numeric", "Outlier detection", "One sample", "Approximately normal", "Grubbs test", "For one suspected outlier only"],
    ["After ANOVA, find exactly which groups differ", "Numeric", "Pairwise post-hoc", "Independent groups", "ANOVA significant", "Tukey HSD", "Controls family-wise error rate"],
], columns=[
    "Business Problem",
    "Outcome Type",
    "Comparison Type",
    "Sample Design",
    "Assumption / Condition",
    "Recommended Test",
    "Business Note"
])

pd.set_option("display.max_colwidth", 120)
decision_table

,Business Problem,Outcome Type,Comparison Type,Sample Design,Assumption / Condition,Recommended Test,Business Note
0,Average order value changed from target,Numeric,One sample vs fixed value,Independent,Approximately normal,one-sample t-test,Use z-test if population std is known or sample is very large
1,Average IQ / payload / sales vs known claim,Numeric,One sample vs fixed value,Independent,Population std known,z-test,Usually used in textbook/business claim problems
2,Employee score before vs after training,Numeric,Two related measurements,Paired,Differences normal,paired t-test,Same people measured twice
3,Calcium level before vs after treatment,Numeric,Two related measurements,Paired,Differences non-normal,Wilcoxon signed-rank,Non-parametric alternative to paired t-test
4,Sales of two independent regions,Numeric,Two groups,Independent,Normal,independent t-test,Use Welch version if variances are unequal
5,Ratings of two app versions,Ordinal/Numeric,Two groups,Independent,Non-normal,Mann-Whitney U,Use Brunner-Munzel if shapes/variances differ strongly
6,Stress levels across three departments,Numeric,More than two groups,Independent,Normal,one-way ANOVA,Follow with Tukey HSD if ANOVA is significant
7,Customer spend across three stores,Skewed numeric,More than two groups,Independent,Non-normal,Kruskal-Wallis or Mood's Median,Kruskal compares ranks; Mood focuses on medians
8,Same patients measured at three time points,Numeric,More than two related measurements,Paired,Non-normal,Friedman test,Non-parametric repeated-measures test
9,Effect of city and offer type on sales,Numeric,Two categorical factors,Independent,Normal,two-way ANOVA,Tests main effects and interaction


## Gold Rules for Choosing Tests

### Rule 1: First identify the outcome variable

| Outcome variable type | Examples | Test family |
| --- | --- | --- |
| Numeric continuous | revenue, delivery time, weight, salary | t-test, ANOVA, Wilcoxon, Mann-Whitney, Kruskal-Wallis |
| Binary | converted/not converted, churned/not churned | proportion test, McNemar, Cochran's Q |
| Categorical counts | city, gender, product type, payment mode | chi-square, Fisher's exact |
| Ordinal/ranked | satisfaction rank, rating 1-5, priority level | Mann-Whitney, Kruskal-Wallis, Kendall Tau |
| Time-ordered sequence | SLA pass/fail over time | Runs test |

### Rule 2: Decide whether samples are paired or independent

| Situation | Design |
| --- | --- |
| Same customer before and after campaign | Paired |
| Same employee before and after training | Paired |
| Same transaction scored by old and new model | Paired |
| Different customers in control and treatment | Independent |
| Different cities, stores, or departments | Usually independent |

### Rule 3: Parametric vs non-parametric

Use parametric tests when data is reasonably normal, numeric, and assumptions are
acceptable. Use non-parametric tests when data is skewed, ordinal, outlier-heavy,
or the sample is small.

## Choosing Between Commonly Confused Tests

### t-test vs ANOVA

| Use case | Test |
| --- | --- |
| Compare 2 means | t-test |
| Compare 3 or more means | ANOVA |
| Compare 3 or more means and then identify pairs | ANOVA followed by Tukey HSD |

### Paired t-test vs Independent t-test

| Use case | Test |
| --- | --- |
| Same people/items measured twice | Paired t-test |
| Different people/items in two groups | Independent t-test |

### Chi-square vs Fisher's Exact

| Use case | Test |
| --- | --- |
| Large enough categorical table | Chi-square |
| Small 2 x 2 table or low expected counts | Fisher's exact |

### Mann-Whitney vs Wilcoxon

| Use case | Test |
| --- | --- |
| Two independent non-normal groups | Mann-Whitney U |
| Two paired non-normal measurements | Wilcoxon signed-rank |

### Kruskal-Wallis vs Friedman

| Use case | Test |
| --- | --- |
| More than two independent non-normal groups | Kruskal-Wallis |
| More than two related non-normal measurements | Friedman |

### Levene vs Bartlett vs Fligner

| Use case | Test |
| --- | --- |
| Normal data, compare variances | Bartlett |
| General robust variance comparison | Levene |
| Skewed data or outliers | Fligner-Killeen |

## Scenario-Based Mini Flowcharts

### Numeric Outcome

```text
Numeric target
 |
 |-- One sample vs target value
 |      |-- std known / large sample -> z-test
 |      |-- std unknown ------------> one-sample t-test
 |
 |-- Two groups
 |      |-- paired?
 |      |      |-- normal differences -> paired t-test
 |      |      |-- non-normal --------> Wilcoxon
 |      |
 |      |-- independent?
 |             |-- normal ------------> independent t-test
 |             |-- non-normal --------> Mann-Whitney U / Brunner-Munzel
 |
 |-- More than two groups
        |-- paired?
        |      |-- non-normal --------> Friedman
        |
        |-- independent?
               |-- normal ------------> ANOVA
               |-- non-normal --------> Kruskal-Wallis / Mood's Median
```

### Categorical / Binary Outcome

```text
Categorical or binary target
 |
 |-- One category distribution vs expected split -> chi-square goodness-of-fit
 |-- Two categorical variables -----------------> chi-square independence
 |-- Small 2 x 2 table -------------------------> Fisher's exact
 |-- Same subjects before-after binary ---------> McNemar
 |-- Same subjects, 3+ binary treatments -------> Cochran's Q
 |-- One binary rate vs claim ------------------> one-proportion z-test
```

### Assumption and Diagnostic Testing

```text
Need to validate assumptions?
 |
 |-- Normality of raw data ----------> Shapiro-Wilk / KS / Anderson-Darling
 |-- Normality of model residuals ---> D'Agostino-Pearson / Jarque-Bera
 |-- Equal variance -----------------> Levene / Bartlett / Fligner-Killeen
 |-- One suspicious outlier ----------> Grubbs
 |-- Randomness in time sequence ----> Runs test
```

## Industry Examples: Which Test Should I Use?

In [2]:
examples = pd.DataFrame([
    ["Did the new checkout page improve conversion rate?", "Binary conversion rate", "Two independent groups", "Two-proportion z-test if covered; otherwise compare proportions carefully"],
    ["Did training improve the same employees' productivity?", "Numeric score", "Before-after same employees", "Paired t-test; Wilcoxon if non-normal"],
    ["Do three stores have different average bill values?", "Numeric spend", "Three independent groups", "ANOVA; Kruskal-Wallis if skewed"],
    ["Is payment method associated with customer segment?", "Categorical vs categorical", "Independent records", "Chi-square independence"],
    ["Does a small treatment group recover more often?", "Binary outcome", "Small 2 x 2 table", "Fisher's exact"],
    ["Does the new fraud model make different errors?", "Correct/incorrect", "Same transactions", "McNemar"],
    ["Are ratings from three app versions different?", "Ordinal rating", "Three independent groups", "Kruskal-Wallis"],
    ["Do customer satisfaction rank and renewal rank move together?", "Ranked variables", "Same customers", "Kendall Tau"],
    ["Is one city less predictable in delivery time?", "Numeric spread", "Independent city groups", "Levene or Fligner-Killeen"],
    ["Is one fill-volume reading unusually high?", "Numeric measurement", "One suspected outlier", "Grubbs test"],
], columns=["Business Question", "Data", "Design", "Recommended Test"])

examples

,Business Question,Data,Design,Recommended Test
0,Did the new checkout page improve conversion rate?,Binary conversion rate,Two independent groups,Two-proportion z-test if covered; otherwise compare proportions carefully
1,Did training improve the same employees' productivity?,Numeric score,Before-after same employees,Paired t-test; Wilcoxon if non-normal
2,Do three stores have different average bill values?,Numeric spend,Three independent groups,ANOVA; Kruskal-Wallis if skewed
3,Is payment method associated with customer segment?,Categorical vs categorical,Independent records,Chi-square independence
4,Does a small treatment group recover more often?,Binary outcome,Small 2 x 2 table,Fisher's exact
5,Does the new fraud model make different errors?,Correct/incorrect,Same transactions,McNemar
6,Are ratings from three app versions different?,Ordinal rating,Three independent groups,Kruskal-Wallis
7,Do customer satisfaction rank and renewal rank move together?,Ranked variables,Same customers,Kendall Tau
8,Is one city less predictable in delivery time?,Numeric spread,Independent city groups,Levene or Fligner-Killeen
9,Is one fill-volume reading unusually high?,Numeric measurement,One suspected outlier,Grubbs test


## Practical Checklist Before You Run the Test

Use this checklist in real projects:

| Step | Question | Example |
| --- | --- | --- |
| 1 | What is the target variable? | delivery time, conversion, rating |
| 2 | Is the target numeric or categorical? | delivery time is numeric, conversion is binary |
| 3 | How many groups are compared? | two cities or five stores |
| 4 | Are records paired? | same customer before-after or different customers |
| 5 | Is the metric normally distributed? | check histogram and normality test |
| 6 | Are variances similar? | Levene or Fligner-Killeen |
| 7 | Is the sample small? | use exact test for small categorical data |
| 8 | What is the effect size or business impact? | p-value alone is not enough |
| 9 | What decision will be taken? | launch campaign, recalibrate machine, retrain model |

## Final Business Interpretation Template

Use this template after every test:

```text
Business question:
We wanted to test whether __________________________.

Null hypothesis:
There is no significant difference / association / effect in __________________.

Alternative hypothesis:
There is a significant difference / association / effect in __________________.

Test used:
We used __________________ because the outcome was ________, the groups were
________, and the sample design was ________.

Result:
The p-value was ________.

Conclusion:
Since p-value is < / >= 0.05, we reject / fail to reject H0.
In business terms, this means __________________________.

Action:
The recommended next step is __________________________.
```

## Key Takeaway

The correct test is selected by the structure of the problem:

```text
Outcome type + number of groups + paired/independent design + assumptions
= correct hypothesis test
```

Do not start from the test name. Start from the business question and the data
structure.